# PROYECTO INTELIGENCIA ARTIFICIAL
##### INTEGRANTES


*  Andrea Mendoza CI 30.032.234

*   Rosxana Medero CI 29.969.634

*   Samuel Velasquez CI 29.966.921
   





In [ ]:
!pip install scikit-learn==1.8.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 53.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
# ==========================================
# FASE 1: CARGA Y EXPLORACIÓN INICIAL
# ==========================================
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Cargar el dataset shop.csv
df = pd.read_csv('/content/dataset shop.csv')

# 2. Exploración de los 10 atributos numéricos y 8 categóricos
df.info()

print("\nResumen estadístico de variables numéricas:")
display(df.describe())

# 3. Aislar la variable objetivo: 'Revenue'
y = df['Revenue']
X = df.drop('Revenue', axis=1)

print(f"\nDimensión de las características de entrada (X): {X.shape}")
print(f"Dimensión de la variable objetivo (y): {y.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427,2.124006,2.357097,3.147364,4.069586
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917,0.911325,1.717277,2.401591,4.025169
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000



Dimensión de las características de entrada (X): (12330, 17)
Dimensión de la variable objetivo (y): (12330,)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# 1. Identificar automáticamente las columnas numéricas y categóricas
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'bool']).columns

# 2. Crear los pipelines de transformación para cada tipo de dato
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Imputa nulos con la mediana
    ('scaler', StandardScaler())                   # Escala los datos numéricos
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Imputa nulos con el más frecuente
    ('onehot', OneHotEncoder(handle_unknown='ignore'))    # Convierte texto/bool a números
])

# 3. Unir los transformadores en un preprocesador usando ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Pipeline de preprocesamiento configurado con éxito.")

Pipeline de preprocesamiento configurado con éxito.


In [ ]:
# ==========================================
# FASE 3 Y 4: AUMENTO DE DATOS Y ENTRENAMIENTO
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# 1. Dividir los datos (80% entrenamiento, 20% validación)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Crear el pipeline completo (Preprocesamiento -> SMOTE -> Modelo)
# Usamos RandomForestClassifier porque es excelente para datos tabulares y nos ayudará a llegar al 90%
pipeline_completo = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)), # Aumento de datos
    ('classifier', RandomForestClassifier(random_state=42))
])

# 3. Entrenar el modelo con los datos de prueba
pipeline_completo.fit(X_train, y_train)

# 4. Validación inicial
y_pred = pipeline_completo.predict(X_test)
acc_inicial = accuracy_score(y_test, y_pred)

print(f"Accuracy inicial del modelo: {acc_inicial * 100:.2f}%")

Accuracy inicial del modelo: 88.69%


In [ ]:
# ==========================================
# FASE 5: OPTIMIZACIÓN (HYPERPARAMETER TUNING)
# ==========================================
from sklearn.model_selection import GridSearchCV

# 1. Definir los parámetros que queremos afinar del RandomForest
# (Usamos el prefijo 'classifier__' porque está dentro del pipeline)
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_split': [2, 5],
    'classifier__class_weight': ['balanced', None]
}

# 2. Configurar la búsqueda en cuadrícula (GridSearchCV)
grid_search = GridSearchCV(pipeline_completo, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# 3. Ejecutar el entrenamiento afinado (Esto puede tomar un par de minutos)
print("Iniciando búsqueda de hiperparámetros...")
grid_search.fit(X_train, y_train)

# 4. Evaluar el mejor modelo encontrado
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
acc_final = accuracy_score(y_test, y_pred_best)

print(f"Mejores hiperparámetros encontrados:\n{grid_search.best_params_}")
print(f"Accuracy final optimizado: {acc_final * 100:.2f}%")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_best))

Iniciando búsqueda de hiperparámetros...
Mejores hiperparámetros encontrados:
{'classifier__class_weight': 'balanced', 'classifier__max_depth': None, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Accuracy final optimizado: 88.69%

Reporte de Clasificación:
              precision    recall  f1-score   support

       False       0.94      0.93      0.93      2084
        True       0.63      0.66      0.64       382

    accuracy                           0.89      2466
   macro avg       0.78      0.80      0.79      2466
weighted avg       0.89      0.89      0.89      2466



In [ ]:
# ==========================================
# FASE 6: EXPORTACIÓN DEL MODELO
# ==========================================

# 1. Guardar el modelo optimizado en un archivo
nombre_archivo = 'shopping_indicator_model.pkl'
joblib.dump(best_model, nombre_archivo)

print(f"¡Modelo exportado con éxito como '{nombre_archivo}'!")

¡Modelo exportado con éxito como 'shopping_indicator_model.pkl'!


In [ ]:
# ==========================================
# FASE 5: OPTIMIZACIÓN (HYPERPARAMETER TUNING)
#(Opcion 2)
# ==========================================
from sklearn.model_selection import GridSearchCV

# 1. Definir los parámetros que queremos afinar del RandomForest
# (Usamos el prefijo 'classifier__' porque está dentro del pipeline)
# Ampliando la red de búsqueda
param_grid = {
    'classifier__n_estimators': [100, 300, 500], # Probamos con más árboles
    'classifier__max_depth': [None, 15, 25, 30], # Diferentes profundidades
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__criterion': ['gini', 'entropy'] # Diferentes formas de medir la calidad de la división
}

# 2. Configurar la búsqueda en cuadrícula (GridSearchCV)
grid_search = GridSearchCV(pipeline_completo, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# 3. Ejecutar el entrenamiento afinado (Esto puede tomar un par de minutos)
print("Iniciando búsqueda de hiperparámetros...")
grid_search.fit(X_train, y_train)

# 4. Evaluar el mejor modelo encontrado
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
acc_final = accuracy_score(y_test, y_pred_best)

print(f"Mejores hiperparámetros encontrados:\n{grid_search.best_params_}")
print(f"Accuracy final optimizado: {acc_final * 100:.2f}%")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_best))

Iniciando búsqueda de hiperparámetros...
Mejores hiperparámetros encontrados:
{'classifier__criterion': 'entropy', 'classifier__max_depth': 25, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 300}
Accuracy final optimizado: 88.89%

Reporte de Clasificación:
              precision    recall  f1-score   support

       False       0.94      0.93      0.93      2084
        True       0.63      0.69      0.66       382

    accuracy                           0.89      2466
   macro avg       0.79      0.81      0.80      2466
weighted avg       0.89      0.89      0.89      2466



In [ ]:
# 1. Guardar el modelo optimizado en un archivo
nombre_archivo = 'shopping_indicator_model_V1.pkl'
joblib.dump(best_model, nombre_archivo)

print(f"¡Modelo exportado con éxito como '{nombre_archivo}'!")

¡Modelo exportado con éxito como 'shopping_indicator_model_V1.pkl'!
